In [1]:
import math 
import numpy as np 
import matplotlib.pyplot as py
%matplotlib inline 

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data 
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label 
    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad 
            other. grad += 1.0 * out.grad 
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data, (self,other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad 
        out._backward = _backward 
        return out
        
   
    
    def __pow__(self, other): 
        assert isinstance(other, (int, float)), "only supoorting int/float powers"
        out = Value(self.data ** other, (self, ), f'**{other}')
        def _backward():
            self.grad += other * (self ** (other-1)) * out.grad 
        out._backward = _backward
        return out 
        
    def __rmul__(self, other): # other * self 
        return self * other 
        
    def __radd__(self, other): # other + self
        return self + other 
        


    def __truediv__(self, other): 
        return self * (other ** -1)

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
        
    def tanh(self):
        x = self.data
        n = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(n, (self, ), 'tanh')
        def _backward():
            self.grad += (1 - n**2) * out.grad 
        out._backward = _backward
        return out 
        

    def exp(self):
        x = self.data 
        n = math.exp(x)
        out = Value(n, (self, ), 'exp')
        def _backward():
            self.grad = out.data * out.grad 
        out._backward = _backward
        return out 

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
        
        


In [ ]:
a = Value(3.0, label = 'a')
b = Value(2.0, label = 'b')
e = a * b; e.label = 'e'
c = Value(-2.0, label = 'c')
d = e + c; d.label = 'd'
d

In [ ]:
import os
os.environ["PATH"] += os.pathsep + "/opt/homebrew/bin"

In [ ]:
!pip install graphviz

In [ ]:
import graphviz

In [ ]:
# to visualize our backprop staretgy 
from graphviz import Digraph 
def trace(root):
    #builds set of all nodes n edges in a grpah
    nodes,edges=set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot=Digraph(format='svg',graph_attr={'rankdir':'LR'}) #LR=left to right

    nodes,edges=trace(root)
    for n in nodes:
        uid=str(id(n))
        dot.node(name=uid,label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=uid+n._op,label=n._op)
            dot.edge(uid+n._op,uid)
    for n1,n2 in edges:
        dot.edge(str(id(n1)),str(id(n2)) + n2._op)
    return dot 
    

In [ ]:

d.backward()
draw_dot(d)


In [ ]:
# plug in the small value h in different variables and notice what change happens in the output when you nudge them even slightly 
# calulate the grad wrt that 
def gradCalc():

    h = 0.01
    
    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10, label='c')
    e = a*b; e.label = 'e'
    d = e+c;  d.label = 'd'
    f = Value(-2.0, label='f')
    L = d*f
    L1=L.data

    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10, label='c')
    e = a*b; e.label = 'e'
    d = e+c;  d.label = 'd'
    f = Value(-2.0, label='f')
    L = d*f
    L2=L.data

    print((L2-L1)/h)
gradCalc()
        

In [ ]:
x1=Value(2.0,label='x1')
x2=Value(0.0,label='x2')
w1=Value(-3.0,label='w1')
w2=Value(1.0,label='w2')
#bias
b=Value(6.8813735870195432,label='b')
x1w1=x1*w1; x1w1.label='x1w1'
x2w2=x2*w2; x2w2.label='x2w2'
x1w1x2w2=x1w1+x2w2; x1w1x2w2.label='x1w1 + x2w2'
n=x1w1x2w2 + b; n.label='n'
o=n.tanh()
o.backward()


In [ ]:
draw_dot(o)

In [ ]:
# to compare our values to pytorch calculation 
import torch

In [ ]:
x1 = torch.Tensor([2.0]).double() ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double() ; x2.requires_grad=True
w1 = torch.Tensor([-3.0]).double() ; w1.requires_grad=True
w2 = torch.Tensor([1.0]).double() ; w2.requires_grad=True

b = torch.Tensor([6.8813735870195432]).double() ; b.requires_grad=True
n = x1 * w1 + x2 * w2 + b 
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('x1', x1.grad.item())
print('w1', w1.grad.item())
print('x2', x2.grad.item())
print('w2', w2.grad.item())

In [ ]:
import random 

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
    def __call__(self, x):
        act = self.b
        for wi, xi in zip(self.w, x):
            act = act + wi * xi
        out = act.tanh()
        return out
    

In [ ]:
x = [1.0, 2.0]
n = Neuron(2)
n(x)